## Load data

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

# from sklearn.compose import ColumnTransformer
# from sklearn.impute import SimpleImputer
# from sklearn.linear_model import Ridge, RidgeCV
# from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# from sklearn.model_selection import GridSearchCV, KFold, cross_val_predict
# from sklearn.pipeline import Pipeline
# from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
sns.set_theme(style="whitegrid", context="notebook")

In [ ]:
ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

SOCAL_PATH = ROOT / "data" / "SoCal.csv"
REFERENCE_PATH = ROOT / "data" / "market_score_reference.csv"

In [ ]:
data_dictionary = pd.read_csv(ROOT / "data" / "data_dictionary.csv")

In [ ]:
socal = pd.read_csv(SOCAL_PATH, dtype={"Zip Code": "string"})
market_score = pd.read_csv(REFERENCE_PATH, dtype={"Zip Code": "string"})

market_score = market_score.dropna(subset=["Market Score"]).copy()

In [ ]:
socal["PERIOD_BEGIN"] = pd.to_datetime(socal["PERIOD_BEGIN"], errors="coerce")
socal["PERIOD_END"] = pd.to_datetime(socal["PERIOD_END"], errors="coerce")

### latest period

(Source: from project scope pdf)


The dataset was last updated on 2026-01-22. However, not all ZIP codes have market feature records for 2025 Q4 (PERIOD_END in December 2025). 

We therefore define a ZIP’s “latest-period” features as the most recent available record within calendar year 2025 (i.e., the row with the maximum PERIOD_END where YEAR(PERIOD_END)=2025).

ZIP codes with no 2025 records should be flagged (e.g., has_2025 = false) and excluded from any “latest-period” scoring output. If any such ZIP codes appear in the reference score table, they should also be excluded from model calibration to avoid misalignment.

In [ ]:
latest_period_lkp = socal[['Zip Code', 'PERIOD_END']].groupby('Zip Code').max().reset_index()
latest_period_lkp['end_year_2025'] = (latest_period_lkp['PERIOD_END'].dt.year == 2025)
latest_period_lkp = latest_period_lkp.set_index("Zip Code")["end_year_2025"]
socal["has_2025"] = socal["Zip Code"].map(latest_period_lkp)

In [ ]:
# max year of PERIOD_END is 2025
socal[['Zip Code', 'PERIOD_END']].groupby('Zip Code').max().reset_index()['PERIOD_END'].dt.year.max()

 Create two new columns to identify latest period and mapped scores

In [ ]:
# Previous logic retained for reference; it flagged every 2025 row as latest.
# socal['latest_period'] = (socal['PERIOD_END'].dt.year == 2025) & (socal["has_2025"] == True)

# Latest period: one most recent available record within calendar year 2025 per ZIP.
socal["latest_period"] = False
rows_in_2025 = socal["PERIOD_END"].dt.year.eq(2025)
latest_2025_indices = (
    socal.loc[rows_in_2025]
    .groupby("Zip Code")["PERIOD_END"]
    .idxmax()
)
socal.loc[latest_2025_indices, "latest_period"] = True

market_score["Market_Score"] = pd.to_numeric(market_score["Market Score"], errors="coerce")
score_map = market_score.set_index("Zip Code")["Market_Score"]
socal["Market_Score"] = socal["Zip Code"].map(score_map)

### [NOTE] `has_2025` vs. `latest_period`

These variables serve different purposes and should not be interchangeable.

#### Definitions

- **`has_2025`**: ZIP-level eligibility flag. It is `True` for every row belonging to a ZIP that has at least one record in 2025.
- **`latest_period`**: Row-level selection flag. It is `True` only for the ZIP’s most recent observation within 2025.

A ZIP can therefore have many rows with `has_2025 = True`, but normally only one row with `latest_period = True`.

#### Modeling use

Use `latest_period = True` to:

- Match market features to the reference score.
- Train and validate the required model.
- Generate the latest-period scoring output.

Use `has_2025` to:

- Identify eligible ZIPs.
- Exclude ZIPs without 2025 data.
- Report data coverage.
- Construct the latest-period dataset.

#### Interpretation of the reference score

The reference score is best treated as a **latest-2025 score with long-term characteristics**:

- It is point-in-time calibrated because the project explicitly associates it with each ZIP’s latest 2025 record.
- It has through-the-cycle qualities because the score should remain relatively stable and reflect persistent market strength.

The missing timestamp in the reference file does not mean the score should be assigned to every historical observation.

#### Why historical rows should not receive the same score

Assigning the reference score to every row with `has_2025 = True` would:

- Artificially multiply the labeled sample.
- Give ZIPs with longer histories more weight.
- Match historical conditions to a 2025 score.
- Create train-validation leakage.
- Imply that scores never change over time.

#### Recommended approach

```text
Identify ZIPs with 2025 data
        ↓
Select each ZIP’s maximum 2025 PERIOD_END
        ↓
Join reference scores to those latest rows
        ↓
Train on labeled latest-period ZIPs
        ↓
Predict scores for unlabeled latest-period ZIPs
```

Historical observations can still provide long-term averages, trends, volatility measures, stability testing, and distribution checks. They should inform the predictors rather than create artificial historical labels.

## Data Cleaning

1. identify columns that has same values uniformly across all rows and remove these columns
2. identify categorical varibles and encode them

In [ ]:
# Work on a copy so the original `socal` DataFrame remains available for audit.
socal_clean = socal.copy()

# 1. Remove columns with one unique value across all rows.
# dropna=False treats a column containing only missing values as uniform as well.
unique_counts = socal_clean.nunique(dropna=False)
uniform_columns = unique_counts[unique_counts <= 1].index.tolist()

uniform_column_summary = pd.DataFrame({
    "Column": uniform_columns,
    "Uniform value": [
        socal_clean[column].iloc[0] if len(socal_clean) else np.nan
        for column in uniform_columns
    ],
})
display(Markdown("### Uniform columns removed"))
display(uniform_column_summary)

socal_clean = socal_clean.drop(columns=uniform_columns)

# 2. Identify categorical variables after uniform-column removal.
# Dates stay as datetime columns. ZIP is retained as a join/reporting identifier,
# but is not one-hot encoded because it should not be a model predictor.
categorical_columns = socal_clean.select_dtypes(
    include=["object", "string", "category", "bool"]
).columns.tolist()
identifier_columns = [column for column in ["Zip Code"] if column in categorical_columns]
categorical_feature_columns = [
    column for column in categorical_columns if column not in identifier_columns
]

categorical_summary = pd.DataFrame({
    "Column": categorical_columns,
    "Unique values (including missing)": [
        socal_clean[column].nunique(dropna=False) for column in categorical_columns
    ],
    "Treatment": [
        "Identifier — retained, not encoded"
        if column in identifier_columns else "One-hot encoded"
        for column in categorical_columns
    ],
})
display(Markdown("### Categorical variables and treatment"))
display(categorical_summary)

# One-hot encode nominal variables. Keep every category for EDA and create an
# explicit missing-category indicator. A later regression pipeline may use
# drop_first=True or an encoder fitted only on training data.
socal_encoded = pd.get_dummies(
    socal_clean,
    columns=categorical_feature_columns,
    prefix=categorical_feature_columns,
    prefix_sep="=",
    dummy_na=True,
    drop_first=False,
    dtype="int8",
)

encoding_check = pd.DataFrame({
    "Dataset": ["Before cleaning", "After uniform-column removal", "After encoding"],
    "Rows": [len(socal), len(socal_clean), len(socal_encoded)],
    "Columns": [socal.shape[1], socal_clean.shape[1], socal_encoded.shape[1]],
})
display(Markdown("### Cleaning and encoding check"))
display(encoding_check)

assert not socal_clean.nunique(dropna=False).le(1).any()
assert socal_encoded.shape[0] == socal.shape[0]
assert not set(categorical_feature_columns).intersection(socal_encoded.columns)

In [ ]:
socal_clean['City'].unique()

In [ ]:
len(socal['Market_Score'].unique())

[Continue ..]

3. identify columns with missing data, and caluclate 1) percentage of missingness of the entire dataset, 2) percentage of missingness for data with column latest_period = True3) percentage of missingness for data with a market score. 4) percentage of missingness for data with a market score and latest_period = True; summarize these four statistics. 
4. since there are more than 300 cities and 52 unique market score, let's Drop the column 'City', and edit categorical_columns accordingly
5. create a list of numerical columns and call it 'numerical_columns'; make sure that 'Market_score' is not included.


In [ ]:
# 3. Compare missingness across all data and the three requested subsets.
latest_period_mask = socal_clean["latest_period"].eq(True)
latest_period_data = socal_clean.loc[latest_period_mask]
market_score_data = socal_clean.loc[socal_clean["Market_Score"].notna()]
market_score_latest_data = socal_clean.loc[
    latest_period_mask & socal_clean["Market_Score"].notna()
]

if latest_period_data.empty:
    raise ValueError("No rows have latest_period = True. Check how latest_period was defined.")
if market_score_data.empty:
    raise ValueError("No rows have a nonmissing Market_Score. Check the score mapping.")
if market_score_latest_data.empty:
    raise ValueError("No rows have both a market score and latest_period = True.")

missingness_summary = pd.DataFrame({
    "Missing count - all data": socal_clean.isna().sum(),
    "Missing % - all data": socal_clean.isna().mean().mul(100),
    "Missing count - latest period": latest_period_data.isna().sum(),
    "Missing % - latest period": latest_period_data.isna().mean().mul(100),
    "Missing count - with market score": market_score_data.isna().sum(),
    "Missing % - with market score": market_score_data.isna().mean().mul(100),
    "Missing count - score and latest period": market_score_latest_data.isna().sum(),
    "Missing % - score and latest period": market_score_latest_data.isna().mean().mul(100),
})

# Keep only columns with missing data in at least one of the four populations.
missingness_summary = (
    missingness_summary.loc[
        lambda frame: frame[[
            "Missing count - all data",
            "Missing count - latest period",
            "Missing count - with market score",
            "Missing count - score and latest period",
        ]]
        .gt(0)
        .any(axis=1)
    ]
    .rename_axis("Column")
    .sort_values("Missing % - latest period", ascending=False)
)

missingness_overview = pd.DataFrame({
    "Population": [
        "All cleaned data",
        "latest_period = True",
        "Market_Score is available",
        "Market_Score available and latest_period = True",
    ],
    "Rows": [
        len(socal_clean), len(latest_period_data), len(market_score_data), len(market_score_latest_data)
    ],
    "Columns with missing data": [
        socal_clean.isna().any().sum(),
        latest_period_data.isna().any().sum(),
        market_score_data.isna().any().sum(),
        market_score_latest_data.isna().any().sum(),
    ],
    "Missing cells": [
        socal_clean.isna().sum().sum(),
        latest_period_data.isna().sum().sum(),
        market_score_data.isna().sum().sum(),
        market_score_latest_data.isna().sum().sum(),
    ],
    "Missing cells as % of population": [
        socal_clean.isna().to_numpy().mean() * 100,
        latest_period_data.isna().to_numpy().mean() * 100,
        market_score_data.isna().to_numpy().mean() * 100,
        market_score_latest_data.isna().to_numpy().mean() * 100,
    ],
})

display(Markdown("### Missingness overview"))
display(missingness_overview.style.format({
    "Rows": "{:,.0f}",
    "Columns with missing data": "{:,.0f}",
    "Missing cells": "{:,.0f}",
    "Missing cells as % of population": "{:.2f}%",
}))
display(Markdown("### Missingness by column"))
display(missingness_summary.style.format({
    "Missing count - all data": "{:,.0f}",
    "Missing % - all data": "{:.2f}%",
    "Missing count - latest period": "{:,.0f}",
    "Missing % - latest period": "{:.2f}%",
    "Missing count - with market score": "{:,.0f}",
    "Missing % - with market score": "{:.2f}%",
    "Missing count - score and latest period": "{:,.0f}",
    "Missing % - score and latest period": "{:.2f}%",
}))

# 4. Remove City and refresh the categorical-variable lists.
try:
    socal_clean = socal_clean.drop(columns="City")
except:
    print("City is not available in socal_clean.")

categorical_columns = socal_clean.select_dtypes(
    include=["object", "string", "category", "bool"]
).columns.tolist()
identifier_columns = [column for column in ["Zip Code"] if column in categorical_columns]
categorical_feature_columns = [
    column for column in categorical_columns if column not in identifier_columns
]

# Zip Code is an identifier, while has_2025 and latest_period are eligibility/selection flags. None should be predictors.
categorical_columns = [col for col in categorical_columns if col not in ["Zip Code", "has_2025", "latest_period"]]

# Refresh the encoded dataset so it no longer contains City dummy variables.
socal_encoded = pd.get_dummies(
    socal_clean,
    columns=categorical_feature_columns,
    prefix=categorical_feature_columns,
    prefix_sep="=",
    dummy_na=True,
    drop_first=False,
    dtype="int8",
)

categorical_summary_after_city_drop = pd.DataFrame({
    "Column": categorical_columns,
    "Unique values (including missing)": [
        socal_clean[column].nunique(dropna=False) for column in categorical_columns
    ],
    "Treatment": [
        "Identifier - retained, not encoded"
        if column in identifier_columns else "One-hot encoded"
        for column in categorical_columns
    ],
})
display(Markdown("### Categorical variables after dropping City"))
display(categorical_summary_after_city_drop)

# 5. List numerical columns and exclude the market-score target.
target_columns = {"Market_Score", "Market_score"}
numerical_columns = [
    column
    for column in socal_clean.select_dtypes(include=[np.number]).columns
    if column not in target_columns
]

# update numerical columns to also agree with data dictionary
numerical_columns = [col for col in numerical_columns if col in set(data_dictionary.loc[data_dictionary["Column_type"].eq("Numerical"), "Variable"])]

display(Markdown("### Numerical predictor candidates"))
display(pd.DataFrame({"Column": numerical_columns}))

assert "City" not in socal_clean.columns
assert "City" not in categorical_columns
assert not any(column.startswith("City=") for column in socal_encoded.columns)
assert target_columns.isdisjoint(numerical_columns)

In [ ]:
missingness_overview.to_csv(ROOT / "analysis" / "missingness_overview.csv")
missingness_summary.to_csv(ROOT / "analysis" / "missingness_summary.csv")

In [ ]:
categorical_columns

In [ ]:
numerical_columns

In [ ]:
socal_clean.columns

### Export cleaned data

In [ ]:
socal_clean.to_csv(ROOT / "data" / "SoCal_clean.csv")

## Exploration

1. create a dataset that contains only data from the latest period and market score is available; these will be use to build the scoring model; and as confirmed above, all data falls in this category has no missingness.
2. create visualization to display relations of scores vs numerical variales (use scatter plot with a linear regression fitted line), and scores vs categorical varibales (use box plot); display these plots in 3x3 matrics.


In [ ]:
# Use the existing selection flag and mapped score without creating new columns.
modeling_data = (
    socal_clean.loc[
        socal_clean["latest_period"].eq(True)
        & socal_clean["Market_Score"].notna()
    ]
    .sort_values(["Zip Code", "PERIOD_END"])
    .reset_index(drop=True)
)

# Confirm that the selected calibration sample follows the requested filters.
modeling_data_summary = pd.DataFrame({
    "Measure": [
        "Rows",
        "Unique ZIP codes",
        "Earliest selected PERIOD_END",
        "Latest selected PERIOD_END",
        "Columns with missing data",
        "Missing cells",
    ],
    "Value": [
        len(modeling_data),
        modeling_data["Zip Code"].nunique(),
        modeling_data["PERIOD_END"].min(),
        modeling_data["PERIOD_END"].max(),
        int(modeling_data.isna().any().sum()),
        int(modeling_data.isna().sum().sum()),
    ],
})
display(Markdown("### Latest-period labeled modeling sample"))
display(modeling_data_summary)

assert modeling_data["latest_period"].eq(True).all()
assert modeling_data["Market_Score"].notna().all()
assert modeling_data.isna().sum().sum() == 0

In [ ]:
# Explort Modeling Data
modeling_data.to_csv(ROOT / "data" / "modeling_data.csv")

In [ ]:
# Plot 

from pathlib import Path
from scipy.stats import norm

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
PLOTS_DIR = PROJECT_ROOT / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

scores = modeling_data["Market_Score"].dropna()
x = np.linspace(scores.min(), scores.max(), 300)
mu, sigma = norm.fit(scores)

display(Markdown("### Empirical Distribution of Market Scores"))

plt.figure(figsize=(9, 5))
sns.histplot(scores, bins=15, stat="density", color="#4C72B0",
             edgecolor="white", alpha=0.65, label="Observed scores")
plt.plot(x, norm.pdf(x, mu, sigma), color="#C44E52", linewidth=2.5,
         label=f"Normal fit: μ={mu:.1f}, σ={sigma:.1f}")

plt.title("Empirical Distribution of Market Scores")
plt.xlabel("Market Score")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.savefig(PLOTS_DIR / "market_score_empirical_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
PLOTS_DIR = PROJECT_ROOT / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)


def plot_numeric_relationships(data, columns, target="Market_Score"):
    """Display 3x3 pages and save both pages and individual numerical plots."""
    for start in range(0, len(columns), 9):
        page_columns = columns[start:start + 9]
        fig, axes = plt.subplots(3, 3, figsize=(17, 14))
        axes = axes.ravel()

        for axis, column in zip(axes, page_columns):
            sns.regplot(
                data=data,
                x=column,
                y=target,
                ci=95,
                scatter_kws={"alpha": 0.65, "s": 30},
                line_kws={"color": "#C44E52", "linewidth": 2},
                ax=axis,
            )
            correlation = data[[column, target]].corr(method="spearman").iloc[0, 1]
            axis.set_title(f"{column}\nSpearman r = {correlation:.2f}", fontsize=10)
            axis.set_xlabel(column, fontsize=9)
            axis.set_ylabel(target, fontsize=9)
            axis.tick_params(labelsize=8)

            individual_fig, individual_axis = plt.subplots(figsize=(8, 6))
            sns.regplot(
                data=data, x=column, y=target, ci=95,
                scatter_kws={"alpha": 0.65, "s": 30},
                line_kws={"color": "#C44E52", "linewidth": 2},
                ax=individual_axis,
            )
            individual_axis.set_title(
                f"{column} vs. Market Score\nSpearman r = {correlation:.2f}"
            )
            individual_axis.set_xlabel(column)
            individual_axis.set_ylabel(target)
            individual_fig.tight_layout()
            individual_fig.savefig(
                PLOTS_DIR / f"{column}_vs_score.png", dpi=300, bbox_inches="tight"
            )
            plt.close(individual_fig)

        for axis in axes[len(page_columns):]:
            axis.set_visible(False)

        page_number = start // 9 + 1
        total_pages = max(1, int(np.ceil(len(columns) / 9)))
        fig.suptitle(
            f"Market Score vs. numerical variables ({page_number}/{total_pages})",
            fontsize=15,
            y=1.01,
        )
        plt.tight_layout()
        fig.savefig(
            PLOTS_DIR / f"numerical_variables_vs_score_page_{page_number}.png",
            dpi=300, bbox_inches="tight",
        )
        plt.show()

def plot_categorical_relationships(data, columns, target="Market_Score"):
    """Display 3x3 pages and save both pages and individual categorical plots."""
    for start in range(0, len(columns), 9):
        page_columns = columns[start:start + 9]
        fig, axes = plt.subplots(3, 3, figsize=(18, 14))
        axes = axes.ravel()

        for axis, column in zip(axes, page_columns):
            category_order = (
                data.groupby(column, observed=True)[target]
                .median()
                .sort_values()
                .index
                .tolist()
            )
            sns.boxplot(
                data=data,
                x=column,
                y=target,
                order=category_order,
                color="#4C72B0",
                showfliers=True,
                ax=axis,
            )
            axis.set_title(column, fontsize=10)
            axis.set_xlabel(column, fontsize=9)
            axis.set_ylabel(target, fontsize=9)
            axis.tick_params(axis="x", rotation=45, labelsize=8)
            axis.tick_params(axis="y", labelsize=8)

            individual_fig, individual_axis = plt.subplots(figsize=(9, 6))
            sns.boxplot(
                data=data, x=column, y=target, order=category_order,
                color="#4C72B0", showfliers=True, ax=individual_axis,
            )
            individual_axis.set_title(f"{column} vs. Market Score")
            individual_axis.set_xlabel(column)
            individual_axis.set_ylabel(target)
            individual_axis.tick_params(axis="x", rotation=45)
            individual_fig.tight_layout()
            individual_fig.savefig(
                PLOTS_DIR / f"{column}_vs_score.png", dpi=300, bbox_inches="tight"
            )
            plt.close(individual_fig)

        for axis in axes[len(page_columns):]:
            axis.set_visible(False)

        page_number = start // 9 + 1
        total_pages = max(1, int(np.ceil(len(columns) / 9)))
        fig.suptitle(
            f"Market Score vs. categorical variables ({page_number}/{total_pages})",
            fontsize=15,
            y=1.01,
        )
        plt.tight_layout()
        fig.savefig(
            PLOTS_DIR / f"categorical_variables_vs_score_page_{page_number}.png",
            dpi=300, bbox_inches="tight",
        )
        plt.show()

display(Markdown("### Market Score vs. numerical variables"))
plot_numeric_relationships(modeling_data, numerical_columns)

display(Markdown("### Market Score vs. categorical variables"))
plot_categorical_relationships(modeling_data, categorical_columns)

display(pd.DataFrame({
    "Variable type": ["Numerical", "Categorical"],
    "Variables plotted": [len(numerical_columns), len(categorical_columns)],
    "3x3 figure pages": [
        int(np.ceil(len(numerical_columns) / 9)),
        int(np.ceil(len(categorical_columns) / 9)),
    ],
}))

Notes:
1. consider log transformation on some highly skewed data
2. consider reduce the dimentionality of some categorical variables
3. consider to change some numerical data to categorical data

In [ ]:
# Export features

import json

feature_lists = {
    "numerical_columns": numerical_columns,
    "categorical_columns": categorical_columns,
}

with open(ROOT / "data" / "model_feature_lists.json", "w", encoding="utf-8") as file:
    json.dump(feature_lists, file, indent=4)

## Test Dataset Creation

Now that modeling_data (which include only data that latest_peiord = True) will be used to do modeling, let's create following test dataset with same data manipulation as modeling_data for calibration and full-period scoring. Start from social_clean, create
1) has_2025_data dataset that include all data has_2025=True for latest_period calibration (one thought is calibrate based on the mean value of each calendar year and by shifting the predicted score by annual mean difference)
2) full_peiord dataset that includes all data and apply the same manipulation as social_clean


In [ ]:
# [Code Here]
# Both scoring datasets start from socal_clean so they inherit the same
# cleaning and column structure as modeling_data.
has_2025_data = (
    socal_clean.loc[socal_clean["has_2025"].eq(True)]
    .sort_values(["Zip Code", "PERIOD_END"])
    .reset_index(drop=True)
    .copy()
)

full_period_data = (
    socal_clean
    .sort_values(["Zip Code", "PERIOD_END"])
    .reset_index(drop=True)
    .copy()
)

# Confirm that filtering changed rows only, not the feature schema.
assert list(has_2025_data.columns) == list(modeling_data.columns)
assert list(full_period_data.columns) == list(modeling_data.columns)
assert has_2025_data["has_2025"].eq(True).all()
assert len(full_period_data) == len(socal_clean)

test_dataset_summary = pd.DataFrame({
    "Dataset": ["modeling_data", "has_2025_data", "full_period_data"],
    "Rows": [len(modeling_data), len(has_2025_data), len(full_period_data)],
    "Unique ZIP codes": [
        modeling_data["Zip Code"].nunique(),
        has_2025_data["Zip Code"].nunique(),
        full_period_data["Zip Code"].nunique(),
    ],
    "Earliest PERIOD_END": [
        modeling_data["PERIOD_END"].min(),
        has_2025_data["PERIOD_END"].min(),
        full_period_data["PERIOD_END"].min(),
    ],
    "Latest PERIOD_END": [
        modeling_data["PERIOD_END"].max(),
        has_2025_data["PERIOD_END"].max(),
        full_period_data["PERIOD_END"].max(),
    ],
    "Rows with reference score": [
        modeling_data["Market_Score"].notna().sum(),
        has_2025_data["Market_Score"].notna().sum(),
        full_period_data["Market_Score"].notna().sum(),
    ],
    "Missing cells": [
        modeling_data.isna().sum().sum(),
        has_2025_data.isna().sum().sum(),
        full_period_data.isna().sum().sum(),
    ],
})

display(Markdown("### Calibration and full-period scoring datasets"))
display(test_dataset_summary.style.format({
    "Rows": "{:,.0f}",
    "Unique ZIP codes": "{:,.0f}",
    "Rows with reference score": "{:,.0f}",
    "Missing cells": "{:,.0f}",
}))

# Annual coverage is shown for evaluating a future time-adjustment approach.
# No score shift is applied here because historical annual target means are unavailable.
annual_coverage_summary = (
    has_2025_data.groupby(has_2025_data["PERIOD_END"].dt.year)
    .agg(
        rows=("Zip Code", "size"),
        unique_zips=("Zip Code", "nunique"),
        rows_with_reference_score=("Market_Score", "count"),
    )
    .rename_axis("Calendar year")
    .reset_index()
)
display(Markdown("### Annual coverage within has_2025_data"))
display(annual_coverage_summary)

In [ ]:
# Explort Test Data
has_2025_data.to_csv(ROOT / "data" / "has_2025_data.csv")
full_period_data.to_csv(ROOT / "data" / "full_period_data.csv")